# 02 - Contrato y reglas de calidad

Esta notebook convierte reglas de calidad en verificaciones ejecutables. Sigue la Clase 08: el Gobierno del Dato asigna decisiones y responsabilidades; no es solamente código. También retoma las dimensiones y métricas de la Clase 07.

El ejemplo es local, ficticio y autocontenido. Las funciones se definen aquí mismo, cada línea de código tiene un comentario didáctico y las celdas se ejecutan de arriba hacia abajo. Usaremos pandas y la biblioteca estándar, sin depender de `gobierno_demo.py` ni servicios externos.

## 1. Configuración del entorno y dependencias

Usamos pandas para tablas y módulos de Python para JSON, fechas, expresiones regulares y pruebas. La semilla y la fecha de corte son fijas: así los resultados se repiten al volver a ejecutar la clase. No se necesitan descargas ni paquetes de validación adicionales.

In [1]:
import json  # Importa JSON para serializar el contrato y el reporte.
import random  # Importa el generador aleatorio para fijar una semilla reproducible.
import re  # Importa expresiones regulares para validar emails.
from datetime import date, datetime, timezone  # Importa tipos de fecha y zona horaria.
from io import StringIO  # Permite leer el CSV de ejemplo desde texto.
import pandas as pd  # Importa pandas para cargar y analizar datos tabulares.

random.seed(42)  # Fija la semilla para que las pruebas aleatorias sean reproducibles.
ZONA_HORARIA = timezone.utc  # Define UTC como zona horaria de referencia.
FORMATO_FECHA = "%Y-%m-%d"  # Define el formato de fecha aceptado en el archivo.
FECHA_CORTE = pd.Timestamp("2025-01-14", tz="UTC")  # Fija la fecha de corte para revisar actualidad.
MAX_ANTIGUEDAD_DIAS = 30  # Define el máximo de días de antigüedad para este ejemplo.
MAX_RECHAZO = 0.20  # Define el máximo porcentaje de filas rechazadas permitido.
print("pandas:", pd.__version__)  # Muestra la versión de pandas activa.
print("Fecha de corte UTC:", FECHA_CORTE)  # Confirma la fecha reproducible del análisis.

pandas: 3.0.2
Fecha de corte UTC: 2025-01-14 00:00:00+00:00


## 2. Carga de datos de entrada y dataset de prueba

Para que cualquier estudiante pueda repetir el ejemplo sin buscar archivos, leeremos un CSV pequeño desde texto con `read_csv`. Incluye casos válidos e inválidos. Las fechas se convierten explícitamente; una fecha imposible se conserva como `NaT` para que una regla la detecte, no para corregirla en silencio.

In [3]:
csv_prueba = "\n".join([  # Construye un archivo CSV reproducible desde texto.
    "id_venta,fecha_venta,id_cliente,id_producto,monto,moneda,cantidad,fecha_actualizacion,email",  # Define las columnas de entrada.
    "1,2025-01-05,101,P01,1500,ARS,1,2025-01-06,a@b.com",  # Incluye una venta válida.
    "2,2025-01-06,102,P02,2300, ars ,2,2025-01-07,juan@ok.com",  # Incluye una moneda con espacios y minúsculas.
    "2,2025-01-06,102,P02,2300,ars,2,2025-01-07,juan@ok.com",  # Repite el ID para probar unicidad.
    "3,2025-99-01,103,P01,850,ARS,1,2025-01-10,ana@test.com",  # Incluye una fecha imposible.
    ",2025-01-08,104,P01,1200,ARS,1,2025-01-09,ana2@test.com",  # Deja vacío un identificador obligatorio.
    "5,2025-01-09,9999,P02,-50,USD,1,2025-01-10,cliente@test.com",  # Incluye un cliente desconocido y un monto negativo.
    "6,2025-01-10,106,P01,0,EUR,0,2025-01-10,c@test.com",  # Incluye monto y cantidad fuera de regla.
    "7,2025-01-11,107,P02,150000,BTC,1,2025-01-12,juan@@err.com",  # Incluye monto extremo, moneda inválida y email mal formado.
    "8,2025-01-12,108,P01,1800,ARS,1,2025-01-13,b@test.com",  # Incluye otra venta válida.
    "9,2025-01-13,109,P02,2000,USD,2,2025-01-14,c@test.com",  # Incluye monto distinto a su referencia de exactitud.
    "10,2025-01-14,110,P02,1600,ARS,1,2025-01-14,d@test.com",  # Incluye una venta válida en la fecha de corte.
    "11,2025-01-15,111,P01,1600,ARS,1,2025-01-15,e@test.com",  # Incluye una fecha futura respecto de la fecha de corte.
])  # Termina el texto del archivo CSV.
df_crudo = pd.read_csv(StringIO(csv_prueba))  # Lee el CSV con pandas.
df = df_crudo.copy()  # Conserva el origen y crea una tabla para analizar.
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"], format=FORMATO_FECHA, errors="coerce", utc=True)  # Convierte fechas inválidas en NaT.
df["fecha_actualizacion"] = pd.to_datetime(df["fecha_actualizacion"], format=FORMATO_FECHA, errors="coerce", utc=True)  # Convierte las fechas de actualización a UTC.
df["moneda"] = df["moneda"].astype("string").str.strip().str.upper()  # Normaliza el formato de moneda sin cambiar el origen.
clientes = pd.DataFrame({"id_cliente": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111]})  # Crea el maestro de clientes ficticio.
productos = pd.DataFrame({"id_producto": ["P01", "P02"]})  # Crea el maestro de productos ficticio.
referencia_montos = {1: 1500.0, 9: 2100.0}  # Crea una referencia parcial para ilustrar exactitud proxy.
print("Filas cargadas:", len(df), "| Clientes de referencia:", len(clientes), "| Productos:", len(productos))  # Resume los datos de entrada.
display(df)  # Muestra los datos que serán evaluados.

Filas cargadas: 12 | Clientes de referencia: 11 | Productos: 2


,id_venta,fecha_venta,id_cliente,id_producto,monto,moneda,cantidad,fecha_actualizacion,email
0,1.0,2025-01-05 00:00:00+00:00,101,P01,1500,ARS,1,2025-01-06 00:00:00+00:00,a@b.com
1,2.0,2025-01-06 00:00:00+00:00,102,P02,2300,ARS,2,2025-01-07 00:00:00+00:00,juan@ok.com
2,2.0,2025-01-06 00:00:00+00:00,102,P02,2300,ARS,2,2025-01-07 00:00:00+00:00,juan@ok.com
3,3.0,NaT,103,P01,850,ARS,1,2025-01-10 00:00:00+00:00,ana@test.com
4,NaN,2025-01-08 00:00:00+00:00,104,P01,1200,ARS,1,2025-01-09 00:00:00+00:00,ana2@test.com
5,5.0,2025-01-09 00:00:00+00:00,9999,P02,-50,USD,1,2025-01-10 00:00:00+00:00,cliente@test.com
6,6.0,2025-01-10 00:00:00+00:00,106,P01,0,EUR,0,2025-01-10 00:00:00+00:00,c@test.com
7,7.0,2025-01-11 00:00:00+00:00,107,P02,150000,BTC,1,2025-01-12 00:00:00+00:00,juan@@err.com
8,8.0,2025-01-12 00:00:00+00:00,108,P01,1800,ARS,1,2025-01-13 00:00:00+00:00,b@test.com
9,9.0,2025-01-13 00:00:00+00:00,109,P02,2000,USD,2,2025-01-14 00:00:00+00:00,c@test.com


## 3. Data profiling para dimensiones de calidad

Medimos completitud, unicidad, validez, consistencia, actualidad y exactitud proxy. La exactitud solo se calcula donde hay una referencia: reportaremos tanto el porcentaje de coincidencia como la cobertura de esa referencia. También inspeccionaremos nulos, duplicados, montos atípicos y dominios observados antes de aplicar las reglas.

In [4]:
def generar_perfil_calidad(tabla):  # Define el perfil de calidad con métricas explícitas.
    total = len(tabla)  # Guarda el denominador del conjunto.
    monedas_normalizadas = df_crudo["moneda"].astype("string").str.strip().str.upper()  # Normaliza monedas solo para medir su dominio.
    fechas_actuales = tabla["fecha_actualizacion"].between(FECHA_CORTE - pd.Timedelta(days=MAX_ANTIGUEDAD_DIAS), FECHA_CORTE)  # Marca actualizaciones no futuras y dentro del plazo.
    filas_con_referencia = tabla[tabla["id_venta"].isin(referencia_montos)]  # Selecciona filas con una referencia de monto.
    coincidencias = sum(fila["monto"] == referencia_montos[fila["id_venta"]] for _, fila in filas_con_referencia.iterrows())  # Cuenta montos que coinciden con la referencia.
    q1 = tabla["monto"].quantile(0.25)  # Calcula el primer cuartil de monto.
    q3 = tabla["monto"].quantile(0.75)  # Calcula el tercer cuartil de monto.
    iqr = q3 - q1  # Calcula el rango intercuartílico.
    limite_inferior = q1 - 1.5 * iqr  # Calcula el límite inferior para outliers.
    limite_superior = q3 + 1.5 * iqr  # Calcula el límite superior para outliers.
    outliers = int((~tabla["monto"].between(limite_inferior, limite_superior)).sum())  # Cuenta montos fuera de los límites IQR.
    perfil = pd.DataFrame([  # Construye una tabla con métricas y denominadores claros.
        {"dimension": "Completitud de id_venta", "resultado": round(tabla["id_venta"].notna().mean() * 100, 1), "denominador": total},  # Mide IDs presentes.
        {"dimension": "Unicidad de id_venta", "resultado": round(tabla["id_venta"].nunique() / total * 100, 1), "denominador": total},  # Mide IDs diferentes por fila.
        {"dimension": "Validez de fecha_venta", "resultado": round(tabla["fecha_venta"].notna().mean() * 100, 1), "denominador": total},  # Mide fechas interpretables.
        {"dimension": "Consistencia de moneda", "resultado": round(monedas_normalizadas.isin(["ARS", "USD", "EUR"]).mean() * 100, 1), "denominador": total},  # Mide monedas dentro del dominio.
        {"dimension": "Actualidad de actualización", "resultado": round(fechas_actuales.mean() * 100, 1), "denominador": total},  # Mide fechas de actualización vigentes.
        {"dimension": "Exactitud proxy", "resultado": round(coincidencias / len(filas_con_referencia) * 100, 1), "denominador": len(filas_con_referencia)},  # Mide coincidencia solo donde hay evidencia.
        {"dimension": "Cobertura de referencia", "resultado": round(len(filas_con_referencia) / total * 100, 1), "denominador": total},  # Mide qué parte del conjunto tiene referencia.
        {"dimension": "Outliers IQR", "resultado": outliers, "denominador": total},  # Cuenta montos que requieren investigación.
    ])  # Termina la tabla de métricas.
    return perfil, (limite_inferior, limite_superior)  # Devuelve métricas y límites IQR para interpretarlos.

perfil, limites_iqr = generar_perfil_calidad(df)  # Ejecuta el perfilado del conjunto cargado.
display(perfil)  # Muestra las métricas por dimensión.
print("Filas completamente duplicadas:", int(df.duplicated().sum()))  # Cuenta filas idénticas en todas las columnas.
print("IDs de venta repetidos:", int(df["id_venta"].duplicated().sum()))  # Cuenta IDs repetidos después de su primera aparición.
print("Límites IQR de monto:", tuple(round(limite, 2) for limite in limites_iqr))  # Muestra el rango de referencia de outliers.
print("Distribución original de moneda:")  # Anuncia la distribución antes de normalizar.
display(df_crudo["moneda"].value_counts(dropna=False).rename("frecuencia").to_frame())  # Muestra monedas y frecuencias del CSV original.
print("Valores nulos por columna:")  # Anuncia el conteo de valores faltantes.
display(df.isna().sum().rename("nulos").to_frame())  # Muestra nulos, incluidas fechas imposibles convertidas en NaT.

,dimension,resultado,denominador
0,Completitud de id_venta,91.7,12
1,Unicidad de id_venta,83.3,12
2,Validez de fecha_venta,91.7,12
3,Consistencia de moneda,91.7,12
4,Actualidad de actualización,91.7,12
5,Exactitud proxy,50.0,2
6,Cobertura de referencia,16.7,12
7,Outliers IQR,1.0,12


Filas completamente duplicadas: 1
IDs de venta repetidos: 1
Límites IQR de monto: (np.float64(-331.25), np.float64(3518.75))
Distribución original de moneda:


,frecuencia
moneda,
ARS,6
USD,2
ars,1
ars,1
EUR,1
BTC,1


Valores nulos por columna:


,nulos
id_venta,1
fecha_venta,1
id_cliente,0
id_producto,0
monto,0
moneda,0
cantidad,0
fecha_actualizacion,0
email,0


## 4. Definición del contrato de datos

El contrato combina esquema y decisiones de negocio. Según la Clase 08, el **Data Owner** aprueba la finalidad y los requisitos; el **Data Steward** mantiene definiciones y reglas; el **Data Engineer** las implementa; el **CDO** define la estrategia y políticas globales. En este ejemplo los roles son ilustrativos y la aprobación está pendiente: el código no reemplaza esa decisión organizativa.

La versión cambia cuando se aprueba una modificación de reglas, no para hacer pasar un lote que falló. El esquema se guarda como diccionario JSON-serializable para inspeccionarlo y versionarlo.

In [5]:
EJECUCION_UTC = datetime.now(ZONA_HORARIA).isoformat()  # Registra cuándo se ejecutó esta demostración.
contrato = {  # Declara el contrato como estructura serializable.
    "nombre": "ventas_para_analitica",  # Identifica el conjunto de datos.
    "version": "1.0.0",  # Marca la versión aprobada del esquema y sus reglas.
    "fecha_vigencia": "2025-01-01",  # Indica desde cuándo aplica esta versión del ejemplo.
    "finalidad": "Analítica de ventas por producto, fecha y moneda",  # Limita el uso previsto del dato.
    "data_owner": "Gerencia Comercial",  # Asigna la decisión de negocio al Owner.
    "data_steward": "Steward Comercial",  # Asigna mantenimiento de definiciones al Steward.
    "data_engineer": "Equipo de Ingeniería de Datos",  # Asigna implementación técnica al equipo.
    "cdo": "Dirección de Datos",  # Registra quién define la estrategia global.
    "estado_aprobacion": "Pendiente de aprobación real",  # Evita presentar el ejemplo como política aprobada.
    "columnas_obligatorias": ["id_venta", "fecha_venta", "id_cliente", "id_producto", "monto", "moneda", "cantidad", "fecha_actualizacion", "email"],  # Define los campos requeridos.
}  # Termina los metadatos principales del contrato.
esquema = {  # Declara tipo, nulabilidad y dominio de cada campo.
    "id_venta": {"tipo": "entero", "nullable": False, "unico": True},  # Define la clave de venta.
    "fecha_venta": {"tipo": "fecha_utc", "nullable": False, "formato": FORMATO_FECHA},  # Define la fecha del evento.
    "id_cliente": {"tipo": "entero", "nullable": False, "referencia": "clientes.id_cliente"},  # Define la relación con clientes.
    "id_producto": {"tipo": "texto", "nullable": False, "referencia": "productos.id_producto"},  # Define la relación con productos.
    "monto": {"tipo": "decimal", "nullable": False, "minimo_exclusivo": 0, "maximo": 50000},  # Define el rango de monto.
    "moneda": {"tipo": "texto", "nullable": False, "dominio": ["ARS", "USD", "EUR"]},  # Define monedas admitidas.
    "cantidad": {"tipo": "entero", "nullable": False, "minimo": 1},  # Define la cantidad mínima.
    "fecha_actualizacion": {"tipo": "fecha_utc", "nullable": False, "formato": FORMATO_FECHA},  # Define la fecha de actualización.
    "email": {"tipo": "texto", "nullable": True, "severidad": "warning"},  # Documenta que el email es opcional para analítica.
}  # Termina la especificación del esquema.
contrato["esquema"] = esquema  # Agrega el esquema al contrato versionado.
print(json.dumps(contrato, ensure_ascii=False, indent=2))  # Serializa e imprime el contrato en formato JSON.

{
  "nombre": "ventas_para_analitica",
  "version": "1.0.0",
  "fecha_vigencia": "2025-01-01",
  "finalidad": "Analítica de ventas por producto, fecha y moneda",
  "data_owner": "Gerencia Comercial",
  "data_steward": "Steward Comercial",
  "data_engineer": "Equipo de Ingeniería de Datos",
  "cdo": "Dirección de Datos",
  "estado_aprobacion": "Pendiente de aprobación real",
  "columnas_obligatorias": [
    "id_venta",
    "fecha_venta",
    "id_cliente",
    "id_producto",
    "monto",
    "moneda",
    "cantidad",
    "fecha_actualizacion",
    "email"
  ],
  "esquema": {
    "id_venta": {
      "tipo": "entero",
      "nullable": false,
      "unico": true
    },
    "fecha_venta": {
      "tipo": "fecha_utc",
      "nullable": false,
      "formato": "%Y-%m-%d"
    },
    "id_cliente": {
      "tipo": "entero",
      "nullable": false,
      "referencia": "clientes.id_cliente"
    },
    "id_producto": {
      "tipo": "texto",
      "nullable": false,
      "referencia": "productos.

## 5. Implementación de reglas de calidad

Aplicaremos reglas de nulabilidad, unicidad, rangos, dominios, regex, integridad referencial y consistencia temporal. Cada regla declara dimensión y severidad: un **error** manda la fila a cuarentena; un **warning** se registra, pero no bloquea por sí solo. El email es opcional porque la finalidad declarada es analítica, no una campaña de marketing.

In [6]:
reglas_contrato = [  # Declara reglas versionadas con dimensión y severidad.
    {"id": "DQ001", "nombre": "id_venta_presente", "dimension": "completitud", "severidad": "error"},  # Exige una clave de venta.
    {"id": "DQ002", "nombre": "id_venta_unico", "dimension": "unicidad", "severidad": "error"},  # Evita claves de venta repetidas.
    {"id": "DQ003", "nombre": "fecha_venta_valida", "dimension": "validez", "severidad": "error"},  # Exige fecha interpretable.
    {"id": "DQ004", "nombre": "fecha_no_futura", "dimension": "consistencia", "severidad": "error"},  # Prohíbe fechas posteriores al corte.
    {"id": "DQ005", "nombre": "cliente_referenciado", "dimension": "integridad", "severidad": "error"},  # Exige cliente existente.
    {"id": "DQ006", "nombre": "producto_referenciado", "dimension": "integridad", "severidad": "error"},  # Exige producto existente.
    {"id": "DQ007", "nombre": "monto_en_rango", "dimension": "exactitud_proxy", "severidad": "error"},  # Exige monto positivo y bajo el máximo didáctico.
    {"id": "DQ008", "nombre": "cantidad_positiva", "dimension": "validez", "severidad": "error"},  # Exige cantidad positiva.
    {"id": "DQ009", "nombre": "moneda_en_dominio", "dimension": "consistencia", "severidad": "error"},  # Exige código de moneda permitido.
    {"id": "DQ010", "nombre": "fecha_actualizacion_vigente", "dimension": "vigencia", "severidad": "error"},  # Exige actualización reciente.
    {"id": "DQ011", "nombre": "email_formato", "dimension": "validez", "severidad": "warning"},  # Advierte sobre formato de email sin bloquear la analítica.
    {"id": "DQ012", "nombre": "monto_contra_referencia", "dimension": "exactitud", "severidad": "warning"},  # Advierte si existe una referencia independiente discrepante.
]  # Termina la definición de las reglas del contrato.
contrato["reglas"] = reglas_contrato  # Asocia reglas y responsabilidades a la versión del contrato.

def validar_esquema(tabla, definicion):  # Define una comprobación estructural del contrato.
    errores = []  # Prepara una lista de diferencias encontradas.
    columnas_requeridas = set(definicion["columnas_obligatorias"])  # Obtiene los nombres obligatorios.
    columnas_recibidas = set(tabla.columns)  # Obtiene los nombres del archivo recibido.
    faltantes = sorted(columnas_requeridas - columnas_recibidas)  # Encuentra columnas ausentes.
    adicionales = sorted(columnas_recibidas - columnas_requeridas)  # Encuentra columnas no declaradas.
    if faltantes:  # Revisa si faltan columnas obligatorias.
        errores.append("Faltan columnas: " + ", ".join(faltantes))  # Registra las columnas que faltan.
    if adicionales:  # Revisa si llegaron columnas desconocidas.
        errores.append("Columnas no declaradas: " + ", ".join(adicionales))  # Registra columnas fuera del contrato.
    if "fecha_venta" in tabla.columns and not pd.api.types.is_datetime64_any_dtype(tabla["fecha_venta"]):  # Comprueba el tipo de fecha de venta.
        errores.append("fecha_venta debe estar convertida a fecha")  # Registra un tipo incompatible.
    return errores  # Devuelve todos los errores estructurales encontrados.

contrato["fecha_ejecucion_utc"] = EJECUCION_UTC  # Guarda la evidencia temporal de esta ejecución.
errores_esquema = validar_esquema(df, contrato)  # Valida nombres de columnas y tipo de fecha.
print("Errores de esquema:", errores_esquema)  # Muestra el resultado de la validación estructural.
assert errores_esquema == []  # Confirma que el CSV de ejemplo respeta la estructura declarada.
df["fila_fuente"] = range(1, len(df) + 1)  # Asigna una referencia estable para rastrear cada fila.

Errores de esquema: []


In [7]:
PATRON_EMAIL = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"  # Define el patrón de formato de email del ejercicio.
DOMINIO_MONEDAS = {"ARS", "USD", "EUR"}  # Define las monedas aceptadas.
IDS_CLIENTE_VALIDOS = set(clientes["id_cliente"])  # Prepara el conjunto de clientes existentes.
IDS_PRODUCTO_VALIDOS = set(productos["id_producto"])  # Prepara el conjunto de productos existentes.
IDS_DUPLICADOS = df["id_venta"].duplicated(keep=False)  # Marca todas las filas que comparten un ID.

def evaluar_reglas(tabla):  # Define la evaluación de calidad fila por fila.
    evidencias = []  # Prepara una lista de resultados trazables.
    for indice, fila in tabla.iterrows():  # Recorre cada registro de entrada.
        id_presente = pd.notna(fila["id_venta"])  # Comprueba que la clave no sea nula.
        fecha_presente = pd.notna(fila["fecha_venta"])  # Comprueba que la fecha se pudo interpretar.
        actualizacion_presente = pd.notna(fila["fecha_actualizacion"])  # Comprueba que exista fecha de actualización.
        id_cliente_presente = pd.notna(fila["id_cliente"])  # Comprueba que exista una clave de cliente.
        id_producto_presente = pd.notna(fila["id_producto"])  # Comprueba que exista una clave de producto.
        moneda_normalizada = str(fila["moneda"]).strip().upper()  # Normaliza el código sin alterar el origen.
        email_presente = pd.notna(fila["email"])  # Comprueba si el email opcional tiene valor.
        email_valido = True if not email_presente else bool(re.match(PATRON_EMAIL, str(fila["email"])))  # Valida formato cuando hay email.
        if fecha_presente:  # Calcula antigüedad solo si la fecha se interpretó.
            antiguedad = FECHA_CORTE - fila["fecha_actualizacion"]  # Compara actualización con la fecha de corte.
            fecha_vigente = pd.Timedelta(0) <= antiguedad <= pd.Timedelta(days=MAX_ANTIGUEDAD_DIAS)  # Comprueba vigencia y evita fechas futuras.
        else:  # Maneja fechas de actualización no interpretables.
            fecha_vigente = None  # Marca la regla como no evaluable.
        if fecha_presente and actualizacion_presente:  # Requiere ambas fechas para comparar orden temporal.
            orden_temporal_valido = fila["fecha_actualizacion"] >= fila["fecha_venta"]  # Comprueba que la actualización no anteceda a la venta.
        else:  # Maneja fechas faltantes o inválidas.
            orden_temporal_valido = None  # Marca la regla temporal como no evaluable.
        tiene_referencia = id_presente and fila["id_venta"] in referencia_montos  # Comprueba si existe evidencia de exactitud.
        if tiene_referencia:  # Compara el monto únicamente cuando hay referencia.
            coincide_referencia = float(fila["monto"]) == referencia_montos[fila["id_venta"]]  # Compara monto recibido con monto de referencia.
        else:  # Maneja registros sin fuente de verdad disponible.
            coincide_referencia = None  # No declara exacto ni incorrecto un dato sin evidencia.
        verificaciones = [  # Reúne todas las reglas que se aplican a esta fila.
            ("DQ001", "completitud", "error", id_presente, fila["id_venta"], "El ID de venta debe estar presente."),  # Revisa la clave obligatoria.
            ("DQ002", "unicidad", "error", None if not id_presente else not IDS_DUPLICADOS.loc[indice], fila["id_venta"], "El ID de venta debe ser único."),  # Revisa claves repetidas.
            ("DQ003", "validez", "error", fecha_presente, fila["fecha_venta"], "La fecha de venta debe ser válida."),  # Revisa fechas imposibles.
            ("DQ004", "consistencia", "error", None if not fecha_presente else fila["fecha_venta"] <= FECHA_CORTE, fila["fecha_venta"], "La fecha de venta no puede ser futura."),  # Revisa fechas posteriores al corte.
            ("DQ005", "integridad", "error", id_cliente_presente and fila["id_cliente"] in IDS_CLIENTE_VALIDOS, fila["id_cliente"], "El cliente debe existir en el maestro."),  # Revisa la clave foránea del cliente.
            ("DQ006", "integridad", "error", id_producto_presente and fila["id_producto"] in IDS_PRODUCTO_VALIDOS, fila["id_producto"], "El producto debe existir en el maestro."),  # Revisa la clave foránea del producto.
            ("DQ007", "exactitud_proxy", "error", pd.notna(fila["monto"]) and 0 < fila["monto"] <= 50000, fila["monto"], "El monto debe ser mayor que cero y no superar 50000."),  # Revisa el rango de negocio del monto.
            ("DQ008", "validez", "error", pd.notna(fila["cantidad"]) and fila["cantidad"] >= 1, fila["cantidad"], "La cantidad debe ser un entero positivo."),  # Revisa la cantidad mínima.
            ("DQ009", "consistencia", "error", moneda_normalizada in DOMINIO_MONEDAS, fila["moneda"], "La moneda debe pertenecer al dominio autorizado."),  # Revisa el dominio de moneda.
            ("DQ010", "vigencia", "error", fecha_vigente, fila["fecha_actualizacion"], "La actualización debe estar dentro de los últimos 30 días."),  # Revisa actualidad del registro.
            ("DQ011", "validez", "warning", email_valido, fila["email"], "El email opcional debe tener formato válido."),  # Registra emails inválidos como advertencia.
            ("DQ012", "exactitud", "warning", coincide_referencia, fila["monto"], "El monto se compara solo si existe referencia."),  # Registra discrepancias como advertencia.
        ]  # Termina las verificaciones para la fila actual.
        for regla, dimension, severidad, cumple, valor, mensaje in verificaciones:  # Recorre resultados de cada regla.
            estado = "No evaluable" if cumple is None else "Cumple" if cumple else "Falla"  # Traduce el resultado a un estado legible.
            evidencias.append({"fila_fuente": int(fila["fila_fuente"]), "id_venta": fila["id_venta"], "regla": regla, "dimension": dimension, "severidad": severidad, "estado": estado, "valor": str(valor), "detalle": mensaje})  # Guarda evidencia trazable.
    return pd.DataFrame(evidencias)  # Devuelve una tabla con una fila por regla evaluada.

## 6. Ejecución del validador de contrato y reglas

Ahora ejecutamos las reglas sobre todas las filas. El detalle conserva aprobaciones, fallas y reglas no evaluables; así no confundimos falta de evidencia con un dato incorrecto. Una fila va a cuarentena si falla al menos una regla `error`. Las advertencias siguen visibles sin cambiar la decisión de aptitud.

In [8]:
detalle_reglas = evaluar_reglas(df)  # Ejecuta todas las reglas definidas para cada fila.
errores_reglas = detalle_reglas[(detalle_reglas["severidad"] == "error") & (detalle_reglas["estado"] == "Falla")]  # Separa fallas que bloquean una fila.
filas_rechazo = set(errores_reglas["fila_fuente"])  # Conserva las filas que fallaron al menos una regla de error.
aceptadas = df[~df["fila_fuente"].isin(filas_rechazo)].copy()  # Conserva las filas sin errores bloqueantes.
rechazadas = df[df["fila_fuente"].isin(filas_rechazo)].copy()  # Separa filas con errores para cuarentena.
advertencias = detalle_reglas[(detalle_reglas["severidad"] == "warning") & (detalle_reglas["estado"] == "Falla")]  # Separa advertencias que no bloquean.

def resumir_reglas(detalle):  # Define el cálculo de cumplimiento por regla.
    resumen = detalle.groupby(["regla", "severidad", "estado"]).size().unstack(fill_value=0)  # Cuenta resultados por regla y estado.
    for estado in ["Cumple", "Falla", "No evaluable"]:  # Recorre los tres estados posibles.
        if estado not in resumen.columns:  # Comprueba si un estado no aparece en esta ejecución.
            resumen[estado] = 0  # Agrega el estado faltante con cero casos.
    resumen["evaluadas"] = resumen["Cumple"] + resumen["Falla"]  # Excluye no evaluables del denominador.
    resumen["cumplimiento_pct"] = (resumen["Cumple"] / resumen["evaluadas"] * 100).round(1)  # Calcula cumplimiento sobre reglas evaluables.
    return resumen.reset_index()  # Devuelve el resumen como una tabla plana.

resumen_reglas = resumir_reglas(detalle_reglas)  # Calcula métricas para cada regla.
print("Resultados por regla:")  # Anuncia el resumen de validación.
display(resumen_reglas)  # Muestra cumplimiento y número de casos no evaluables.
print("Detalle trazable de reglas:")  # Anuncia la evidencia por fila.
display(detalle_reglas)  # Muestra una línea por regla y fila de origen.
print("Filas aptas:", len(aceptadas), "| Filas en cuarentena:", len(rechazadas))  # Resume la separación por errores.
print("Advertencias registradas:", len(advertencias))  # Informa advertencias que no bloquean.

Resultados por regla:


estado,regla,severidad,Cumple,Falla,No evaluable,evaluadas,cumplimiento_pct
0,DQ001,error,11,1,0,12,91.7
1,DQ002,error,9,2,1,11,81.8
2,DQ003,error,11,1,0,12,91.7
3,DQ004,error,10,1,1,11,90.9
4,DQ005,error,11,1,0,12,91.7
5,DQ006,error,12,0,0,12,100.0
6,DQ007,error,9,3,0,12,75.0
7,DQ008,error,11,1,0,12,91.7
8,DQ009,error,11,1,0,12,91.7
9,DQ010,error,10,1,1,11,90.9


Detalle trazable de reglas:


,fila_fuente,id_venta,regla,dimension,severidad,estado,valor,detalle
0,1,1.0,DQ001,completitud,error,Cumple,1.0,El ID de venta debe estar presente.
1,1,1.0,DQ002,unicidad,error,Cumple,1.0,El ID de venta debe ser único.
2,1,1.0,DQ003,validez,error,Cumple,2025-01-05 00:00:00+00:00,La fecha de venta debe ser válida.
3,1,1.0,DQ004,consistencia,error,Cumple,2025-01-05 00:00:00+00:00,La fecha de venta no puede ser futura.
4,1,1.0,DQ005,integridad,error,Cumple,101,El cliente debe existir en el maestro.
...,...,...,...,...,...,...,...,...
139,12,11.0,DQ008,validez,error,Cumple,1,La cantidad debe ser un entero positivo.
140,12,11.0,DQ009,consistencia,error,Cumple,ARS,La moneda debe pertenecer al dominio autorizado.
141,12,11.0,DQ010,vigencia,error,Falla,2025-01-15 00:00:00+00:00,La actualización debe estar dentro de los últi...
142,12,11.0,DQ011,validez,warning,Cumple,e@test.com,El email opcional debe tener formato válido.


Filas aptas: 4 | Filas en cuarentena: 8
Advertencias registradas: 2


## 7. Pruebas automáticas para verificar la ejecución

Las pruebas comprueban el flujo completo y casos de borde: umbral exacto de publicación, lote vacío, rechazo total, deriva de esquema, IDs duplicados en conflicto, exactitud con referencia parcial y advertencias que no bloquean por sí solas. Si una aserción falla, revisamos la regla o el dato; no relajamos el contrato a ciegas.

In [9]:
import unittest  # Importa unittest para ejecutar pruebas sin instalar paquetes adicionales.

def puerta_publicacion(total, rechazadas, max_rechazo=MAX_RECHAZO):  # Define las condiciones para habilitar una carga.
    if not 0 <= max_rechazo <= 1 or total < 0 or not 0 <= rechazadas <= total:  # Detecta parámetros imposibles.
        raise ValueError("Parámetros inválidos para la puerta de publicación")  # Detiene el flujo ante un contrato incoherente.
    if total == 0 or rechazadas == total:  # Bloquea lotes vacíos o sin filas aptas.
        return False  # No publica el lote.
    return rechazadas / total <= max_rechazo  # Compara la tasa de rechazo con el límite aprobado.

publicable = puerta_publicacion(len(df), len(rechazadas))  # Aplica el umbral del 20% al conjunto de prueba.
print("Tasa de rechazo:", round(len(rechazadas) / len(df) * 100, 1), "%")  # Informa la tasa calculada.
print("Estado del lote:", "PUBLICABLE" if publicable else "BLOQUEADO")  # Explica la decisión de la puerta.

class PruebasContrato(unittest.TestCase):  # Agrupa pruebas automáticas del contrato y las reglas.
    def test_resultado_del_conjunto(self):  # Comprueba los totales esperados del ejemplo.
        self.assertEqual(len(df), 12)  # Confirma que se cargaron doce filas.
        self.assertEqual(len(aceptadas), 4)  # Confirma que cuatro filas quedaron aptas.
        self.assertEqual(len(rechazadas), 8)  # Confirma que ocho filas se enviaron a cuarentena.
        self.assertFalse(publicable)  # Confirma que la tasa supera el umbral del contrato.
    def test_limites_de_publicacion(self):  # Comprueba umbrales y casos de borde.
        self.assertTrue(puerta_publicacion(10, 2))  # Acepta exactamente el 20% de rechazo.
        self.assertFalse(puerta_publicacion(10, 3))  # Bloquea una tasa superior al 20%.
        self.assertFalse(puerta_publicacion(0, 0))  # Bloquea un lote vacío.
        self.assertFalse(puerta_publicacion(10, 10))  # Bloquea un lote totalmente rechazado.
    def test_parametros_invalidos(self):  # Comprueba que las entradas incoherentes generan error explícito.
        with self.assertRaises(ValueError):  # Espera un error si hay más rechazos que filas.
            puerta_publicacion(3, 4)  # Ejecuta el caso inválido.
        with self.assertRaises(ValueError):  # Espera un error si el umbral supera el 100%.
            puerta_publicacion(10, 0, max_rechazo=1.1)  # Ejecuta el segundo caso inválido.
    def test_deriva_de_esquema(self):  # Comprueba que una columna obligatoria ausente sea detectada.
        incompleto = df_crudo.drop(columns=["moneda"])  # Crea una copia sin la columna de moneda.
        errores = validar_esquema(incompleto, contrato)  # Valida la copia contra el contrato.
        self.assertTrue(any("moneda" in error for error in errores))  # Confirma que el esquema informa la columna faltante.
    def test_duplicados_en_conflicto(self):  # Comprueba que ambas filas duplicadas se ponen en cuarentena.
        fuentes_duplicadas = set(df.loc[df["id_venta"] == 2, "fila_fuente"])  # Obtiene las filas de origen con ID 2.
        self.assertEqual(fuentes_duplicadas, {2, 3})  # Confirma que se detectan ambas filas del conflicto.
        self.assertTrue(fuentes_duplicadas.issubset(filas_rechazo))  # Confirma que ninguna se elige silenciosamente.
    def test_referencia_parcial_y_advertencia(self):  # Comprueba cobertura de exactitud y severidad warning.
        regla_exactitud = detalle_reglas[detalle_reglas["regla"] == "DQ012"]  # Selecciona resultados de exactitud proxy.
        self.assertEqual((regla_exactitud["estado"] == "No evaluable").sum(), 10)  # Confirma que diez filas no tienen referencia.
        self.assertEqual((regla_exactitud["estado"] == "Falla").sum(), 1)  # Confirma una discrepancia con referencia.
        self.assertIn(10, set(aceptadas["fila_fuente"]))  # Confirma que una advertencia sola no pone en cuarentena.

resultado_pruebas = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(PruebasContrato))  # Ejecuta las seis pruebas.
assert resultado_pruebas.wasSuccessful()  # Falla la celda si alguna prueba no se aprueba.
print("Seis pruebas del contrato aprobadas.")  # Resume el resultado satisfactorio de las pruebas.

test_deriva_de_esquema (__main__.PruebasContrato.test_deriva_de_esquema) ... ok
test_duplicados_en_conflicto (__main__.PruebasContrato.test_duplicados_en_conflicto) ... ok
test_limites_de_publicacion (__main__.PruebasContrato.test_limites_de_publicacion) ... ok
test_parametros_invalidos (__main__.PruebasContrato.test_parametros_invalidos) ... ok
test_referencia_parcial_y_advertencia (__main__.PruebasContrato.test_referencia_parcial_y_advertencia) ... ok
test_resultado_del_conjunto (__main__.PruebasContrato.test_resultado_del_conjunto) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.018s

OK


Tasa de rechazo: 66.7 %
Estado del lote: BLOQUEADO
Seis pruebas del contrato aprobadas.


## 8. Reporte de calidad y artefactos

El reporte separa tres resultados: el esquema del contrato, las reglas aplicadas a los datos y la ejecución de las pruebas. Este lote debe terminar con contrato estructural aprobado, reglas con fallas detectadas y publicación bloqueada; eso demuestra que el validador encuentra problemas, no que el notebook haya fallado. Guardaremos evidencia JSON, el resumen y detalle en CSV, y un resumen HTML.

In [10]:
from pathlib import Path  # Importa rutas para guardar artefactos en la carpeta de la unidad.

perfil_para_reporte = {str(fila["dimension"]): {"resultado": float(fila["resultado"]), "denominador": int(fila["denominador"])} for _, fila in perfil.iterrows()}  # Convierte métricas de pandas a tipos serializables.
fallas_por_regla = {str(regla): int(cantidad) for regla, cantidad in errores_reglas.groupby("regla").size().items()}  # Resume errores bloqueantes por regla.
reporte_final = {  # Construye el estado final del contrato y su ejecución.
    "contrato": {"nombre": contrato["nombre"], "version": contrato["version"], "estado": "PASS" if not errores_esquema else "FAIL", "errores_esquema": errores_esquema},  # Registra el resultado estructural.
    "datos": {"filas_entrada": int(len(df)), "filas_aptas": int(len(aceptadas)), "filas_rechazadas": int(len(rechazadas)), "tasa_rechazo_pct": round(len(rechazadas) / len(df) * 100, 1), "estado_reglas": "FAIL" if not errores_reglas.empty else "PASS", "fallas_por_regla": fallas_por_regla, "advertencias": int(len(advertencias)), "perfil": perfil_para_reporte},  # Resume resultados de calidad con denominadores.
    "publicacion": {"estado": "PASS" if publicable else "FAIL", "motivo": "La tasa de rechazo respeta el umbral." if publicable else "La tasa de rechazo supera el umbral aprobado."},  # Registra la decisión de la puerta.
    "pruebas": {"estado": "PASS" if resultado_pruebas.wasSuccessful() else "FAIL", "pruebas_aprobadas": int(resultado_pruebas.testsRun)},  # Registra la ejecución de pruebas.
    "evidencia": {"fecha_ejecucion_utc": EJECUCION_UTC, "fecha_corte": str(FECHA_CORTE), "archivo_fuente": "CSV ficticio embebido en la notebook"},  # Documenta reproducibilidad y origen.
}  # Termina el objeto del reporte.
reporte_json = json.dumps(reporte_final, ensure_ascii=False, indent=2)  # Serializa el reporte en JSON legible.
carpeta_unidad = Path.cwd() / "notebook" / "unidad_III"  # Propone la carpeta de notebooks al ejecutar desde la raíz.
carpeta_base = carpeta_unidad if carpeta_unidad.exists() else Path.cwd()  # Adapta la ruta si el kernel ya está en la carpeta de la unidad.
carpeta_salida = carpeta_base / "salidas_clase"  # Define dónde guardar evidencias de esta clase.
carpeta_salida.mkdir(parents=True, exist_ok=True)  # Crea la carpeta de salida si todavía no existe.
(carpeta_salida / "reporte_calidad.json").write_text(reporte_json, encoding="utf-8")  # Guarda métricas y estados en JSON.
(carpeta_salida / "resumen_reglas.csv").write_text(resumen_reglas.to_csv(index=False), encoding="utf-8")  # Guarda el resumen por regla en CSV.
(carpeta_salida / "detalle_reglas.csv").write_text(detalle_reglas.to_csv(index=False), encoding="utf-8")  # Guarda trazabilidad por fila y regla en CSV.
(carpeta_salida / "reporte_calidad.html").write_text(resumen_reglas.to_html(index=False), encoding="utf-8")  # Guarda un resumen HTML de las reglas.
print("Contrato:", reporte_final["contrato"]["estado"], "| Reglas:", reporte_final["datos"]["estado_reglas"], "| Publicación:", reporte_final["publicacion"]["estado"], "| Pruebas:", reporte_final["pruebas"]["estado"])  # Presenta los cuatro estados finales.
print("Artefactos guardados en:", carpeta_salida.resolve())  # Muestra la ubicación de los reportes.
print(reporte_json)  # Muestra la evidencia JSON generada.
display(resumen_reglas)  # Muestra el resumen tabular que también se guardó en CSV.
print("Vista previa HTML:", (carpeta_salida / "reporte_calidad.html").read_text(encoding="utf-8")[:250])  # Confirma que el HTML puede leerse.

Contrato: PASS | Reglas: FAIL | Publicación: FAIL | Pruebas: PASS
Artefactos guardados en: C:\Users\orozcos\clases\Ingenieria_Datos\notebook\unidad_III\salidas_clase
{
  "contrato": {
    "nombre": "ventas_para_analitica",
    "version": "1.0.0",
    "estado": "PASS",
    "errores_esquema": []
  },
  "datos": {
    "filas_entrada": 12,
    "filas_aptas": 4,
    "filas_rechazadas": 8,
    "tasa_rechazo_pct": 66.7,
    "estado_reglas": "FAIL",
    "fallas_por_regla": {
      "DQ001": 1,
      "DQ002": 2,
      "DQ003": 1,
      "DQ004": 1,
      "DQ005": 1,
      "DQ007": 3,
      "DQ008": 1,
      "DQ009": 1,
      "DQ010": 1
    },
    "advertencias": 2,
    "perfil": {
      "Completitud de id_venta": {
        "resultado": 91.7,
        "denominador": 12
      },
      "Unicidad de id_venta": {
        "resultado": 83.3,
        "denominador": 12
      },
      "Validez de fecha_venta": {
        "resultado": 91.7,
        "denominador": 12
      },
      "Consistencia de moneda": {


estado,regla,severidad,Cumple,Falla,No evaluable,evaluadas,cumplimiento_pct
0,DQ001,error,11,1,0,12,91.7
1,DQ002,error,9,2,1,11,81.8
2,DQ003,error,11,1,0,12,91.7
3,DQ004,error,10,1,1,11,90.9
4,DQ005,error,11,1,0,12,91.7
5,DQ006,error,12,0,0,12,100.0
6,DQ007,error,9,3,0,12,75.0
7,DQ008,error,11,1,0,12,91.7
8,DQ009,error,11,1,0,12,91.7
9,DQ010,error,10,1,1,11,90.9


Vista previa HTML: <table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>estado</th>
      <th>regla</th>
      <th>severidad</th>
      <th>Cumple</th>
      <th>Falla</th>
      <th>No evaluable</th>
      <th>evaluadas</th>
   
